##Transformação da tabela tips para a camada Silver

In [0]:
from pyspark.sql import functions as F

##Ler tabela bronze

In [0]:
df_trips_bronze = spark.table(
    "workspace.bronze_logistics.trips"
)

##Visuaizar dados

In [0]:
display(df_trips_bronze.limit(10))

In [0]:
df_trips_bronze.printSchema()

In [0]:
display(
    df_trips_bronze
    .filter(
        F.col("driver_id").isNull() |
        F.col("truck_id").isNull() |
        F.col("trailer_id").isNull()
    )
    .groupBy(
        F.col("driver_id").isNull().alias("driver_nulo"),
        F.col("truck_id").isNull().alias("truck_nulo"),
        F.col("trailer_id").isNull().alias("trailer_nulo")
    )
    .count()
    .orderBy(F.desc("count"))
)

##Ler tabela auxiliar de abastecimentos

In [0]:
df_fuel_purchases_bronze = spark.table(
    "workspace.bronze_logistics.fuel_purchases"
)

##Preparar recursos identificados por viagem

In [0]:
recursos_por_trip = (
    df_fuel_purchases_bronze
    .groupBy("trip_id")
    .agg(
        F.countDistinct("driver_id").alias("quantidade_drivers_fuel"),
        F.countDistinct("truck_id").alias("quantidade_trucks_fuel"),

        F.first(
            "driver_id",
            ignorenulls=True
        ).alias("driver_id_fuel"),

        F.first(
            "truck_id",
            ignorenulls=True
        ).alias("truck_id_fuel")
    )
)

##Verificar recursos que podem ser recuperados

In [0]:
display(
    df_trips_bronze
    .join(
        recursos_por_trip,
        on="trip_id",
        how="left"
    )
    .select(
        F.sum(
            F.when(
                F.col("driver_id").isNull()
                & F.col("driver_id_fuel").isNotNull()
                & (F.col("quantidade_drivers_fuel") == 1),
                1
            ).otherwise(0)
        ).alias("drivers_recuperaveis"),

        F.sum(
            F.when(
                F.col("truck_id").isNull()
                & F.col("truck_id_fuel").isNotNull()
                & (F.col("quantidade_trucks_fuel") == 1),
                1
            ).otherwise(0)
        ).alias("trucks_recuperaveis"),

        F.sum(
            F.when(
                F.col("quantidade_drivers_fuel") > 1,
                1
            ).otherwise(0)
        ).alias("trips_com_multiplos_drivers_no_fuel"),

        F.sum(
            F.when(
                F.col("quantidade_trucks_fuel") > 1,
                1
            ).otherwise(0)
        ).alias("trips_com_multiplos_trucks_no_fuel")
    )
)

##Recuperar recursos ausentes por meio dos abastecimentos

In [0]:
df_trips_enriquecido = (
    df_trips_bronze
    .join(
        recursos_por_trip,
        on="trip_id",
        how="left"
    )
    .withColumn(
        "driver_id_recuperado",
        F.when(
            F.col("driver_id").isNull()
            & F.col("driver_id_fuel").isNotNull()
            & (F.col("quantidade_drivers_fuel") == 1),
            True
        ).otherwise(False)
    )
    .withColumn(
        "truck_id_recuperado",
        F.when(
            F.col("truck_id").isNull()
            & F.col("truck_id_fuel").isNotNull()
            & (F.col("quantidade_trucks_fuel") == 1),
            True
        ).otherwise(False)
    )
    .withColumn(
        "driver_id",
        F.when(
            F.col("driver_id").isNull()
            & (F.col("quantidade_drivers_fuel") == 1),
            F.col("driver_id_fuel")
        ).otherwise(F.col("driver_id"))
    )
    .withColumn(
        "truck_id",
        F.when(
            F.col("truck_id").isNull()
            & (F.col("quantidade_trucks_fuel") == 1),
            F.col("truck_id_fuel")
        ).otherwise(F.col("truck_id"))
    )
    .drop(
        "quantidade_drivers_fuel",
        "quantidade_trucks_fuel",
        "driver_id_fuel",
        "truck_id_fuel"
    )
)

##Padronizar colunas textuais

In [0]:
df_trips_padronizado = (
    df_trips_enriquecido
    .withColumn("trip_id", F.trim("trip_id"))
    .withColumn("load_id", F.trim("load_id"))
    .withColumn("driver_id", F.trim("driver_id"))
    .withColumn("truck_id", F.trim("truck_id"))
    .withColumn("trailer_id", F.trim("trailer_id"))
    .withColumn("trip_status", F.initcap(F.trim("trip_status")))
)

##Criar atributos temporais da viagem

In [0]:
df_trips_temporal = (
    df_trips_padronizado
    .withColumn(
        "trip_year",
        F.year("dispatch_date")
    )
    .withColumn(
        "trip_month",
        F.month("dispatch_date")
    )
    .withColumn(
        "trip_quarter",
        F.quarter("dispatch_date")
    )
    .withColumn(
        "trip_year_month",
        F.date_format("dispatch_date", "yyyy-MM")
    )
)

##Criar indicadores de alocação de recursos

In [0]:
df_trips_qualidade = (
    df_trips_temporal
    .withColumn(
        "has_driver",
        F.col("driver_id").isNotNull()
    )
    .withColumn(
        "has_truck",
        F.col("truck_id").isNotNull()
    )
    .withColumn(
        "has_trailer",
        F.col("trailer_id").isNotNull()
    )
    .withColumn(
        "has_complete_resource_assignment",
        F.col("driver_id").isNotNull()
        & F.col("truck_id").isNotNull()
        & F.col("trailer_id").isNotNull()
    )
)

##Criar métricas operacionais derivadas

In [0]:
df_trips_silver = (
    df_trips_qualidade
    .withColumn(
        "calculated_mpg",
        F.when(
            F.col("fuel_gallons_used") > 0,
            F.round(
                F.col("actual_distance_miles")
                / F.col("fuel_gallons_used"),
                2
            )
        )
    )
    .withColumn(
        "average_speed_mph",
        F.when(
            F.col("actual_duration_hours") > 0,
            F.round(
                F.col("actual_distance_miles")
                / F.col("actual_duration_hours"),
                2
            )
        )
    )
    .withColumn(
        "idle_time_percentage",
        F.when(
            F.col("actual_duration_hours") > 0,
            F.round(
                (
                    F.col("idle_time_hours")
                    / F.col("actual_duration_hours")
                ) * 100,
                2
            )
        )
    )
    .withColumn(
        "mpg_difference",
        F.round(
            F.abs(
                F.col("average_mpg")
                - F.col("calculated_mpg")
            ),
            2
        )
    )
)

##Validar preservação do volume de dados

In [0]:
total_bronze = df_trips_bronze.count()
total_silver = df_trips_silver.count()

print(f"Total de registros na Bronze: {total_bronze}")
print(f"Total de registros na Silver: {total_silver}")
print(f"Diferença de registros: {total_silver - total_bronze}")

##Validar chave primária após as transformações

In [0]:
display(
    df_trips_silver.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("trip_id").alias("trip_ids_unicos"),

        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos")
    )
)

##Validar resultado da recuperação de recursos

In [0]:
display(
    df_trips_silver.select(
        F.sum(
            F.when(
                F.col("driver_id_recuperado"),
                1
            ).otherwise(0)
        ).alias("drivers_recuperados"),

        F.sum(
            F.when(
                F.col("truck_id_recuperado"),
                1
            ).otherwise(0)
        ).alias("trucks_recuperados"),

        F.sum(
            F.when(
                F.col("driver_id").isNull(),
                1
            ).otherwise(0)
        ).alias("drivers_ainda_nulos"),

        F.sum(
            F.when(
                F.col("truck_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trucks_ainda_nulos"),

        F.sum(
            F.when(
                F.col("trailer_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trailers_nulos"),

        F.sum(
            F.when(
                F.col("has_complete_resource_assignment"),
                1
            ).otherwise(0)
        ).alias("viagens_com_alocacao_completa")
    )
)

##Validar métricas operacionais calculadas

In [0]:
display(
    df_trips_silver.select(
        F.sum(
            F.when(
                F.col("calculated_mpg") <= 0,
                1
            ).otherwise(0)
        ).alias("mpg_calculado_invalido"),

        F.sum(
            F.when(
                F.col("average_speed_mph") <= 0,
                1
            ).otherwise(0)
        ).alias("velocidade_media_invalida"),

        F.sum(
            F.when(
                (F.col("idle_time_percentage") < 0)
                | (F.col("idle_time_percentage") > 100),
                1
            ).otherwise(0)
        ).alias("percentual_marcha_lenta_invalido"),

        F.max("mpg_difference").alias("maior_diferenca_mpg")
    )
)

##Visualizar resultado da transformação

In [0]:
display(
    df_trips_silver
    .select(
        "trip_id",
        "load_id",
        "driver_id",
        "truck_id",
        "trailer_id",
        "dispatch_date",
        "trip_year_month",
        "actual_distance_miles",
        "actual_duration_hours",
        "fuel_gallons_used",
        "average_mpg",
        "calculated_mpg",
        "average_speed_mph",
        "idle_time_percentage",
        "driver_id_recuperado",
        "truck_id_recuperado",
        "has_complete_resource_assignment",
        "trip_status"
    )
    .limit(20)
)

##Investigar inconsistências no tempo de marcha lenta

In [0]:
display(
    df_trips_silver.select(
        F.sum(
            F.when(
                F.col("idle_time_hours") < 0,
                1
            ).otherwise(0)
        ).alias("tempo_marcha_lenta_negativo"),

        F.sum(
            F.when(
                F.col("idle_time_hours") >
                F.col("actual_duration_hours"),
                1
            ).otherwise(0)
        ).alias("tempo_marcha_lenta_maior_que_duracao"),

        F.max("idle_time_hours").alias("maior_tempo_marcha_lenta"),

        F.max("actual_duration_hours").alias("maior_duracao_viagem")
    )
)

##Visualizar viagens com tempo de marcha lenta inconsistente

In [0]:
display(
    df_trips_silver
    .filter(
        F.col("idle_time_hours") >
        F.col("actual_duration_hours")
    )
    .select(
        "trip_id",
        "dispatch_date",
        "actual_distance_miles",
        "actual_duration_hours",
        "idle_time_hours",
        "idle_time_percentage",
        "driver_id",
        "truck_id"
    )
    .orderBy(F.desc("idle_time_percentage"))
    .limit(30)
)

##Tratar inconsistências na métrica de marcha lenta

In [0]:
df_trips_silver_final = (
    df_trips_silver
    .withColumn(
        "idle_time_inconsistent",
        F.col("idle_time_hours").isNotNull()
        & F.col("actual_duration_hours").isNotNull()
        & (
            F.col("idle_time_hours") >
            F.col("actual_duration_hours")
        )
    )
    .withColumn(
        "idle_time_percentage",
        F.when(
            (F.col("actual_duration_hours") > 0)
            & (F.col("idle_time_hours") >= 0)
            & (
                F.col("idle_time_hours")
                <= F.col("actual_duration_hours")
            ),
            F.round(
                (
                    F.col("idle_time_hours")
                    / F.col("actual_duration_hours")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
)

##Validar tratamento do tempo de marcha lenta

In [0]:
display(
    df_trips_silver_final.select(
        F.sum(
            F.when(
                F.col("idle_time_inconsistent"),
                1
            ).otherwise(0)
        ).alias("viagens_com_tempo_marcha_lenta_inconsistente"),

        F.sum(
            F.when(
                F.col("idle_time_percentage").isNull(),
                1
            ).otherwise(0)
        ).alias("percentuais_marcha_lenta_nulos"),

        F.sum(
            F.when(
                (F.col("idle_time_percentage") < 0)
                | (F.col("idle_time_percentage") > 100),
                1
            ).otherwise(0)
        ).alias("percentuais_marcha_lenta_fora_do_intervalo")
    )
)

##Validar estrutura final da tabela trips Silver

In [0]:
display(
    df_trips_silver_final.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("trip_id").alias("trip_ids_unicos"),

        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos")
    )
)

##Atualizar tabela trips na camada Silver

In [0]:
(
    df_trips_silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver_logistics.trips")
)

##Validar regras de qualidade na tabela gravada

In [0]:
df_trips_silver_gravada = spark.table(
    "workspace.silver_logistics.trips"
)

In [0]:
display(
    df_trips_silver_gravada.select(
        F.sum(
            F.when(
                F.col("idle_time_inconsistent"),
                1
            ).otherwise(0)
        ).alias("viagens_com_marcha_lenta_inconsistente"),

        F.sum(
            F.when(
                F.col("idle_time_percentage").isNull(),
                1
            ).otherwise(0)
        ).alias("percentuais_marcha_lenta_nulos"),

        F.sum(
            F.when(
                (F.col("idle_time_percentage") < 0)
                | (F.col("idle_time_percentage") > 100),
                1
            ).otherwise(0)
        ).alias("percentuais_fora_do_intervalo")
    )
)

##Validar atualização final da tabela trips Silver

In [0]:
df_trips_silver_gravada = spark.table(
    "workspace.silver_logistics.trips"
)

print(
    f"Total de registros gravados: "
    f"{df_trips_silver_gravada.count()}"
)

df_trips_silver_gravada.printSchema()

##Transformação da tabela loads para a camada Silver

##Ler tabela loads da camada Bronze

In [0]:
df_loads_bronze = spark.table(
    "workspace.bronze_logistics.loads"
)

##Visualizar dados da tabela loads

In [0]:
display(
    df_loads_bronze.limit(10)
)

##Verificar schema da tabela loads

In [0]:
df_loads_bronze.printSchema()

##Padronizar colunas textuais da tabela loads

In [0]:
df_loads_padronizado = (
    df_loads_bronze
    .withColumn("load_id", F.trim(F.col("load_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("route_id", F.trim(F.col("route_id")))
    .withColumn("load_type", F.trim(F.col("load_type")))
    .withColumn("load_status", F.trim(F.col("load_status")))
    .withColumn("booking_type", F.trim(F.col("booking_type")))
)

##Criar atributos temporais da carga

In [0]:
df_loads_temporal = (
    df_loads_padronizado
    .withColumn(
        "load_year",
        F.year("load_date")
    )
    .withColumn(
        "load_month",
        F.month("load_date")
    )
    .withColumn(
        "load_quarter",
        F.quarter("load_date")
    )
    .withColumn(
        "load_year_month",
        F.date_format("load_date", "yyyy-MM")
    )
)

##Criar métricas financeiras da carga

In [0]:
df_loads_financeiro = (
    df_loads_temporal
    .withColumn(
        "total_revenue",
        F.round(
            F.col("revenue")
            + F.col("fuel_surcharge")
            + F.col("accessorial_charges"),
            2
        )
    )
    .withColumn(
        "revenue_per_piece",
        F.when(
            F.col("pieces") > 0,
            F.round(
                F.col("revenue") / F.col("pieces"),
                2
            )
        )
    )
    .withColumn(
        "revenue_per_lb",
        F.when(
            F.col("weight_lbs") > 0,
            F.round(
                F.col("revenue") / F.col("weight_lbs"),
                4
            )
        )
    )
)

##Criar indicadores de qualidade da carga

In [0]:
df_loads_silver = (
    df_loads_financeiro
    .withColumn(
        "has_customer",
        F.col("customer_id").isNotNull()
    )
    .withColumn(
        "has_route",
        F.col("route_id").isNotNull()
    )
    .withColumn(
        "has_complete_relationship",
        F.col("customer_id").isNotNull()
        & F.col("route_id").isNotNull()
    )
)

##Validar preservação do volume da tabela loads

In [0]:
total_loads_bronze = df_loads_bronze.count()
total_loads_silver = df_loads_silver.count()

print(f"Total de registros na Bronze: {total_loads_bronze}")
print(f"Total de registros na Silver: {total_loads_silver}")
print(
    f"Diferença de registros: "
    f"{total_loads_silver - total_loads_bronze}"
)

##Validar chave primária da tabela loads Silver

In [0]:
display(
    df_loads_silver.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("load_id").alias("load_ids_unicos"),
        F.sum(
            F.when(
                F.col("load_id").isNull(),
                1
            ).otherwise(0)
        ).alias("load_ids_nulos")
    )
)

##Validar métricas financeiras da tabela loads

In [0]:
display(
    df_loads_silver.select(
        F.sum(
            F.when(
                F.col("total_revenue") < 0,
                1
            ).otherwise(0)
        ).alias("receitas_totais_negativas"),

        F.sum(
            F.when(
                F.col("revenue_per_piece") < 0,
                1
            ).otherwise(0)
        ).alias("receitas_por_peca_negativas"),

        F.sum(
            F.when(
                F.col("revenue_per_lb") < 0,
                1
            ).otherwise(0)
        ).alias("receitas_por_libra_negativas"),

        F.sum(
            F.when(
                (F.col("pieces") > 0)
                & F.col("revenue_per_piece").isNull(),
                1
            ).otherwise(0)
        ).alias("receitas_por_peca_ausentes"),

        F.sum(
            F.when(
                (F.col("weight_lbs") > 0)
                & F.col("revenue_per_lb").isNull(),
                1
            ).otherwise(0)
        ).alias("receitas_por_libra_ausentes")
    )
)

##Validar consistência da receita total

In [0]:
display(
    df_loads_silver.select(
        F.sum(
            F.when(
                F.abs(
                    F.col("total_revenue")
                    - (
                        F.col("revenue")
                        + F.col("fuel_surcharge")
                        + F.col("accessorial_charges")
                    )
                ) > 0.01,
                1
            ).otherwise(0)
        ).alias("receitas_totais_inconsistentes")
    )
)

##Validar indicadores de relacionamento da carga

In [0]:
display(
    df_loads_silver.select(
        F.sum(
            F.when(
                F.col("has_customer"),
                1
            ).otherwise(0)
        ).alias("cargas_com_customer_id"),

        F.sum(
            F.when(
                F.col("has_route"),
                1
            ).otherwise(0)
        ).alias("cargas_com_route_id"),

        F.sum(
            F.when(
                F.col("has_complete_relationship"),
                1
            ).otherwise(0)
        ).alias("cargas_com_relacionamento_completo"),

        F.sum(
            F.when(
                F.col("customer_id").isNull(),
                1
            ).otherwise(0)
        ).alias("customer_ids_nulos"),

        F.sum(
            F.when(
                F.col("route_id").isNull(),
                1
            ).otherwise(0)
        ).alias("route_ids_nulos")
    )
)

##Ler tabelas auxiliares de clientes e rotas

In [0]:
df_customers_bronze = spark.table(
    "workspace.bronze_logistics.customers"
)

df_routes_bronze = spark.table(
    "workspace.bronze_logistics.routes"
)

##Validar relacionamento de loads com customers

In [0]:
loads_sem_customer = (
    df_loads_silver
    .filter(F.col("customer_id").isNotNull())
    .join(
        df_customers_bronze.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Cargas com customer_id não encontrado em customers: "
    f"{loads_sem_customer.count()}"
)

##Validar relacionamento de loads com routes

In [0]:
loads_sem_route = (
    df_loads_silver
    .filter(F.col("route_id").isNotNull())
    .join(
        df_routes_bronze.select("route_id"),
        on="route_id",
        how="left_anti"
    )
)

print(
    "Cargas com route_id não encontrado em routes: "
    f"{loads_sem_route.count()}"
)

##Visualizar resultado final da tabela loads Silver

In [0]:
display(
    df_loads_silver.select(
        "load_id",
        "customer_id",
        "route_id",
        "load_date",
        "load_year_month",
        "load_type",
        "weight_lbs",
        "pieces",
        "revenue",
        "fuel_surcharge",
        "accessorial_charges",
        "total_revenue",
        "revenue_per_piece",
        "revenue_per_lb",
        "load_status",
        "booking_type",
        "has_customer",
        "has_route",
        "has_complete_relationship"
    ).limit(20)
)

##Gravar tabela loads na camada Silver

In [0]:
(
    df_loads_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver_logistics.loads")
)

##Validar gravação da tabela loads Silver

In [0]:
df_loads_silver_gravada = spark.table(
    "workspace.silver_logistics.loads"
)

print(
    "Total de registros gravados na Silver: "
    f"{df_loads_silver_gravada.count()}"
)

df_loads_silver_gravada.printSchema()

##Validar estrutura persistida da tabela loads Silver

In [0]:
display(
    df_loads_silver_gravada.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("load_id").alias("load_ids_unicos"),

        F.sum(
            F.when(
                F.col("load_id").isNull(),
                1
            ).otherwise(0)
        ).alias("load_ids_nulos"),

        F.sum(
            F.when(
                F.col("has_complete_relationship"),
                1
            ).otherwise(0)
        ).alias("cargas_com_relacionamento_completo"),

        F.sum(
            F.when(
                F.col("total_revenue").isNull(),
                1
            ).otherwise(0)
        ).alias("receitas_totais_nulas")
    )
)

##Transformação da tabela Customers para a camada Silver

##Ler tabela customers da camada Bronze

In [0]:
df_customers_bronze = spark.table(
    "workspace.bronze_logistics.customers"
)

##Visualizar dados da tabela customers

In [0]:
display(
    df_customers_bronze.limit(10)
)

##Verificar schema da tabela customers

In [0]:
df_customers_bronze.printSchema()

##Padronizar colunas textuais da tabela customers

In [0]:
df_customers_padronizado = (
    df_customers_bronze
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )
    .withColumn(
        "customer_name",
        F.trim(F.col("customer_name"))
    )
    .withColumn(
        "customer_type",
        F.trim(F.col("customer_type"))
    )
    .withColumn(
        "primary_freight_type",
        F.trim(F.col("primary_freight_type"))
    )
    .withColumn(
        "account_status",
        F.trim(F.col("account_status"))
    )
)

##Criar atributos temporais do contrato

In [0]:
df_customers_temporal = (
    df_customers_padronizado
    .withColumn(
        "contract_start_year",
        F.year("contract_start_date")
    )
    .withColumn(
        "contract_start_month",
        F.month("contract_start_date")
    )
    .withColumn(
        "contract_start_quarter",
        F.quarter("contract_start_date")
    )
    .withColumn(
        "contract_start_year_month",
        F.date_format("contract_start_date", "yyyy-MM")
    )
)

##Criar indicadores de perfil e qualidade do cliente

In [0]:
df_customers_silver = (
    df_customers_temporal
    .withColumn(
        "is_active_customer",
        F.col("account_status") == "Active"
    )
    .withColumn(
        "has_valid_credit_terms",
        F.col("credit_terms_days").isNotNull()
        & (F.col("credit_terms_days") > 0)
    )
    .withColumn(
        "has_positive_revenue_potential",
        F.col("annual_revenue_potential").isNotNull()
        & (F.col("annual_revenue_potential") >= 0)
    )
    .withColumn(
        "has_valid_contract_date",
        F.col("contract_start_date").isNotNull()
        & (F.col("contract_start_date") <= F.current_date())
    )
)

##Validar preservação do volume da tabela customers

In [0]:
total_customers_bronze = df_customers_bronze.count()
total_customers_silver = df_customers_silver.count()

print(f"Total de registros na Bronze: {total_customers_bronze}")
print(f"Total de registros na Silver: {total_customers_silver}")
print(
    "Diferença de registros: "
    f"{total_customers_silver - total_customers_bronze}"
)

##Validar chave primária da tabela customers Silver

In [0]:
display(
    df_customers_silver.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("customer_id").alias("customer_ids_unicos"),

        F.sum(
            F.when(
                F.col("customer_id").isNull(),
                1
            ).otherwise(0)
        ).alias("customer_ids_nulos")
    )
)

##Validar regras numéricas e de data

In [0]:
display(
    df_customers_silver.select(
        F.sum(
            F.when(
                F.col("credit_terms_days") <= 0,
                1
            ).otherwise(0)
        ).alias("prazos_credito_invalidos"),

        F.sum(
            F.when(
                F.col("annual_revenue_potential") < 0,
                1
            ).otherwise(0)
        ).alias("receitas_potenciais_negativas"),

        F.sum(
            F.when(
                F.col("contract_start_date") > F.current_date(),
                1
            ).otherwise(0)
        ).alias("contratos_com_data_futura")
    )
)

##Validar categorias da tabela customers

In [0]:
display(
    df_customers_silver
    .groupBy("customer_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_customers_silver
    .groupBy("primary_freight_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_customers_silver
    .groupBy("account_status")
    .count()
    .orderBy(F.desc("count"))
)

##Validar indicadores de qualidade dos clientes

In [0]:
display(
    df_customers_silver.select(
        F.sum(
            F.when(
                F.col("is_active_customer"),
                1
            ).otherwise(0)
        ).alias("clientes_ativos"),

        F.sum(
            F.when(
                ~F.col("has_valid_credit_terms"),
                1
            ).otherwise(0)
        ).alias("clientes_com_prazo_credito_invalido"),

        F.sum(
            F.when(
                ~F.col("has_positive_revenue_potential"),
                1
            ).otherwise(0)
        ).alias("clientes_com_receita_potencial_invalida"),

        F.sum(
            F.when(
                ~F.col("has_valid_contract_date"),
                1
            ).otherwise(0)
        ).alias("clientes_com_data_contrato_invalida")
    )
)

##Visualizar resultado final da tabela customers Silver

In [0]:
display(
    df_customers_silver.select(
        "customer_id",
        "customer_name",
        "customer_type",
        "credit_terms_days",
        "primary_freight_type",
        "account_status",
        "contract_start_date",
        "contract_start_year_month",
        "annual_revenue_potential",
        "is_active_customer",
        "has_valid_credit_terms",
        "has_positive_revenue_potential",
        "has_valid_contract_date"
    ).limit(20)
)

##Gravar tabela customers na camada Silver

In [0]:
(
    df_customers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver_logistics.customers")
)

##Validar gravação da tabela customers Silver

In [0]:
df_customers_silver_gravada = spark.table(
    "workspace.silver_logistics.customers"
)

print(
    "Total de registros gravados na Silver: "
    f"{df_customers_silver_gravada.count()}"
)

df_customers_silver_gravada.printSchema()

##Validar estrutura persistida da tabela customers Silver

In [0]:
display(
    df_customers_silver_gravada.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("customer_id").alias("customer_ids_unicos"),

        F.sum(
            F.when(
                F.col("customer_id").isNull(),
                1
            ).otherwise(0)
        ).alias("customer_ids_nulos"),

        F.sum(
            F.when(
                F.col("has_valid_credit_terms"),
                1
            ).otherwise(0)
        ).alias("clientes_com_prazo_credito_valido"),

        F.sum(
            F.when(
                F.col("has_positive_revenue_potential"),
                1
            ).otherwise(0)
        ).alias("clientes_com_receita_potencial_valida")
    )
)

##Transformação da tabela Routes para a camada Silver

In [0]:
df_routes_bronze = spark.table(
    "workspace.bronze_logistics.routes"
)

##Visualizar dados da tabela routes

In [0]:
display(
    df_routes_bronze.limit(10)
)

##Verificar schema da tabela routes

In [0]:
df_routes_bronze.printSchema()

##Padronizar colunas textuais da tabela routes

In [0]:
df_routes_padronizado = (
    df_routes_bronze
    .withColumn(
        "route_id",
        F.trim(F.col("route_id"))
    )
    .withColumn(
        "origin_city",
        F.initcap(F.trim(F.col("origin_city")))
    )
    .withColumn(
        "origin_state",
        F.upper(F.trim(F.col("origin_state")))
    )
    .withColumn(
        "destination_city",
        F.initcap(F.trim(F.col("destination_city")))
    )
    .withColumn(
        "destination_state",
        F.upper(F.trim(F.col("destination_state")))
    )
)

##Criar identificadores descritivos da rota

In [0]:
df_routes_descritivo = (
    df_routes_padronizado
    .withColumn(
        "origin_location",
        F.concat_ws(
            " - ",
            F.col("origin_city"),
            F.col("origin_state")
        )
    )
    .withColumn(
        "destination_location",
        F.concat_ws(
            " - ",
            F.col("destination_city"),
            F.col("destination_state")
        )
    )
    .withColumn(
        "route_description",
        F.concat_ws(
            " → ",
            F.col("origin_location"),
            F.col("destination_location")
        )
    )
)

##Criar métricas operacionais e financeiras da rota

In [0]:
df_routes_metricas = (
    df_routes_descritivo
    .withColumn(
        "estimated_rate_per_mile",
        F.round(
            F.col("base_rate_per_mile")
            * (1 + F.col("fuel_surcharge_rate")),
            2
        )
    )
    .withColumn(
        "estimated_route_cost",
        F.round(
            F.col("typical_distance_miles")
            * F.col("base_rate_per_mile")
            * (1 + F.col("fuel_surcharge_rate")),
            2
        )
    )
    .withColumn(
        "average_miles_per_transit_day",
        F.when(
            F.col("typical_transit_days") > 0,
            F.round(
                F.col("typical_distance_miles")
                / F.col("typical_transit_days"),
                2
            )
        )
    )
)

##Criar indicadores de qualidade da rota

In [0]:
df_routes_silver = (
    df_routes_metricas
    .withColumn(
        "has_valid_distance",
        F.col("typical_distance_miles").isNotNull()
        & (F.col("typical_distance_miles") > 0)
    )
    .withColumn(
        "has_valid_base_rate",
        F.col("base_rate_per_mile").isNotNull()
        & (F.col("base_rate_per_mile") > 0)
    )
    .withColumn(
        "has_valid_fuel_surcharge",
        F.col("fuel_surcharge_rate").isNotNull()
        & (F.col("fuel_surcharge_rate") >= 0)
    )
    .withColumn(
        "has_valid_transit_days",
        F.col("typical_transit_days").isNotNull()
        & (F.col("typical_transit_days") > 0)
    )
    .withColumn(
        "has_complete_route",
        F.col("origin_city").isNotNull()
        & F.col("origin_state").isNotNull()
        & F.col("destination_city").isNotNull()
        & F.col("destination_state").isNotNull()
    )
)

##Validar preservação do volume da tabela routes

In [0]:
total_routes_bronze = df_routes_bronze.count()
total_routes_silver = df_routes_silver.count()

print(f"Total de registros na Bronze: {total_routes_bronze}")
print(f"Total de registros na Silver: {total_routes_silver}")
print(
    "Diferença de registros: "
    f"{total_routes_silver - total_routes_bronze}"
)

##Validar chave primária da tabela routes Silver

In [0]:
display(
    df_routes_silver.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("route_id").alias("route_ids_unicos"),

        F.sum(
            F.when(
                F.col("route_id").isNull(),
                1
            ).otherwise(0)
        ).alias("route_ids_nulos")
    )
)

##Validar métricas calculadas da tabela routes

In [0]:
display(
    df_routes_silver.select(
        F.sum(
            F.when(
                F.col("estimated_rate_per_mile") <= 0,
                1
            ).otherwise(0)
        ).alias("tarifas_estimadas_invalidas"),

        F.sum(
            F.when(
                F.col("estimated_route_cost") <= 0,
                1
            ).otherwise(0)
        ).alias("custos_estimados_invalidos"),

        F.sum(
            F.when(
                F.col("average_miles_per_transit_day") <= 0,
                1
            ).otherwise(0)
        ).alias("medias_diarias_invalidas"),

        F.sum(
            F.when(
                F.col("estimated_route_cost").isNull(),
                1
            ).otherwise(0)
        ).alias("custos_estimados_nulos")
    )
)

##Validar consistência do custo estimado da rota

In [0]:
display(
    df_routes_silver.select(
        F.sum(
            F.when(
                F.abs(
                    F.col("estimated_route_cost")
                    - (
                        F.col("typical_distance_miles")
                        * F.col("base_rate_per_mile")
                        * (1 + F.col("fuel_surcharge_rate"))
                    )
                ) > 0.01,
                1
            ).otherwise(0)
        ).alias("custos_estimados_inconsistentes")
    )
)

##Validar indicadores de qualidade das rotas

In [0]:
display(
    df_routes_silver.select(
        F.sum(
            F.when(F.col("has_valid_distance"), 1).otherwise(0)
        ).alias("rotas_com_distancia_valida"),

        F.sum(
            F.when(F.col("has_valid_base_rate"), 1).otherwise(0)
        ).alias("rotas_com_tarifa_valida"),

        F.sum(
            F.when(F.col("has_valid_fuel_surcharge"), 1).otherwise(0)
        ).alias("rotas_com_sobretaxa_valida"),

        F.sum(
            F.when(F.col("has_valid_transit_days"), 1).otherwise(0)
        ).alias("rotas_com_prazo_valido"),

        F.sum(
            F.when(F.col("has_complete_route"), 1).otherwise(0)
        ).alias("rotas_com_localizacao_completa")
    )
)

##Visualizar resultado final da tabela routes Silver

In [0]:
display(
    df_routes_silver.select(
        "route_id",
        "origin_location",
        "destination_location",
        "route_description",
        "typical_distance_miles",
        "base_rate_per_mile",
        "fuel_surcharge_rate",
        "estimated_rate_per_mile",
        "estimated_route_cost",
        "typical_transit_days",
        "average_miles_per_transit_day",
        "has_valid_distance",
        "has_valid_base_rate",
        "has_valid_fuel_surcharge",
        "has_valid_transit_days",
        "has_complete_route"
    ).limit(20)
)

##Gravar tabela routes na camada Silver

In [0]:
(
    df_routes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver_logistics.routes")
)

##Validar gravação da tabela routes Silver

In [0]:
df_routes_silver_gravada = spark.table(
    "workspace.silver_logistics.routes"
)

print(
    "Total de registros gravados na Silver: "
    f"{df_routes_silver_gravada.count()}"
)

df_routes_silver_gravada.printSchema()

##Validar estrutura persistida da tabela routes Silver

In [0]:
display(
    df_routes_silver_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("route_id").alias("route_ids_unicos"),

        F.sum(
            F.when(
                F.col("route_id").isNull(),
                1
            ).otherwise(0)
        ).alias("route_ids_nulos"),

        F.sum(
            F.when(
                F.col("has_complete_route"),
                1
            ).otherwise(0)
        ).alias("rotas_com_localizacao_completa"),

        F.sum(
            F.when(
                F.col("estimated_route_cost").isNull(),
                1
            ).otherwise(0)
        ).alias("custos_estimados_nulos")
    )
)

##Transformação da tabela Drivers para a camada Silver

##Ler tabela drivers da camada Bronze

In [0]:
df_drivers_bronze = spark.table(
    "workspace.bronze_logistics.drivers"
)

##Visualizar dados da tabela drivers

In [0]:
display(
    df_drivers_bronze.limit(10)
)

##Verificar schema da tabela drivers

In [0]:
df_drivers_bronze.printSchema()

##Padronizar colunas textuais da tabela drivers

In [0]:
df_drivers_padronizado = (
    df_drivers_bronze
    .withColumn(
        "driver_id",
        F.trim(F.col("driver_id"))
    )
    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name")))
    )
    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name")))
    )
    .withColumn(
        "license_number",
        F.upper(F.trim(F.col("license_number")))
    )
    .withColumn(
        "license_state",
        F.upper(F.trim(F.col("license_state")))
    )
    .withColumn(
        "home_terminal",
        F.initcap(F.trim(F.col("home_terminal")))
    )
    .withColumn(
        "employment_status",
        F.initcap(F.trim(F.col("employment_status")))
    )
    .withColumn(
        "cdl_class",
        F.upper(F.trim(F.col("cdl_class")))
    )
)

##Criar atributos temporais e de carreira do motorista

In [0]:
df_drivers_temporal = (
    df_drivers_padronizado
    .withColumn(
        "hire_year",
        F.year("hire_date")
    )
    .withColumn(
        "hire_month",
        F.month("hire_date")
    )
    .withColumn(
        "hire_year_month",
        F.date_format("hire_date", "yyyy-MM")
    )
    .withColumn(
        "termination_year",
        F.year("termination_date")
    )
    .withColumn(
        "driver_age",
        F.floor(
            F.months_between(
                F.current_date(),
                F.col("date_of_birth")
            ) / 12
        )
    )
    .withColumn(
        "tenure_years",
        F.round(
            F.months_between(
                F.coalesce(
                    F.col("termination_date"),
                    F.current_date()
                ),
                F.col("hire_date")
            ) / 12,
            2
        )
    )
)

##Criar indicadores de perfil e qualidade do motorista

In [0]:
df_drivers_silver = (
    df_drivers_temporal
    .withColumn(
        "is_active_driver",
        F.col("employment_status") == "Active"
    )
    .withColumn(
        "has_valid_hire_date",
        F.col("hire_date").isNotNull()
        & (F.col("hire_date") <= F.current_date())
    )
    .withColumn(
        "has_valid_birth_date",
        F.col("date_of_birth").isNotNull()
        & (F.col("date_of_birth") < F.col("hire_date"))
    )
    .withColumn(
        "has_valid_experience",
        F.col("years_experience").isNotNull()
        & (F.col("years_experience") >= 0)
    )
    .withColumn(
        "has_valid_termination_date",
        F.col("termination_date").isNull()
        | (
            F.col("hire_date").isNotNull()
            & (F.col("termination_date") >= F.col("hire_date"))
        )
    )
    .withColumn(
        "has_valid_license",
        F.col("license_number").isNotNull()
        & (F.length(F.trim(F.col("license_number"))) > 0)
        & F.col("license_state").isNotNull()
    )
)

##Validar preservação do volume da tabela drivers

In [0]:
total_drivers_bronze = df_drivers_bronze.count()
total_drivers_silver = df_drivers_silver.count()

print(f"Total de registros na Bronze: {total_drivers_bronze}")
print(f"Total de registros na Silver: {total_drivers_silver}")
print(
    "Diferença de registros: "
    f"{total_drivers_silver - total_drivers_bronze}"
)

##Validar chave primária da tabela drivers Silver

In [0]:
display(
    df_drivers_silver.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("driver_id").alias("driver_ids_unicos"),

        F.sum(
            F.when(
                F.col("driver_id").isNull(),
                1
            ).otherwise(0)
        ).alias("driver_ids_nulos")
    )
)

##Validar possíveis duplicidades de licença

In [0]:
display(
    df_drivers_silver
    .groupBy("license_number")
    .count()
    .filter(F.col("count") > 1)
)

##Validar coerência entre status e desligamento

In [0]:
display(
    df_drivers_silver.select(
        F.sum(
            F.when(
                (F.col("employment_status") == "Active")
                & F.col("termination_date").isNotNull(),
                1
            ).otherwise(0)
        ).alias("ativos_com_data_desligamento"),

        F.sum(
            F.when(
                (F.col("employment_status") == "Terminated")
                & F.col("termination_date").isNull(),
                1
            ).otherwise(0)
        ).alias("desligados_sem_data_desligamento")
    )
)

##Validar regras de idade, vínculo e experiência

In [0]:
display(
    df_drivers_silver.select(
        F.sum(
            F.when(
                F.col("driver_age") < 18,
                1
            ).otherwise(0)
        ).alias("motoristas_menores_de_18"),

        F.sum(
            F.when(
                F.col("driver_age") > 80,
                1
            ).otherwise(0)
        ).alias("motoristas_com_idade_superior_a_80"),

        F.sum(
            F.when(
                F.col("tenure_years") < 0,
                1
            ).otherwise(0)
        ).alias("tempos_de_vinculo_negativos"),

        F.sum(
            F.when(
                F.col("years_experience") < 0,
                1
            ).otherwise(0)
        ).alias("experiencias_negativas")
    )
)

##Validar indicadores de qualidade dos motoristas

In [0]:
display(
    df_drivers_silver.select(
        F.sum(
            F.when(
                F.col("is_active_driver"),
                1
            ).otherwise(0)
        ).alias("motoristas_ativos"),

        F.sum(
            F.when(
                ~F.col("has_valid_hire_date"),
                1
            ).otherwise(0)
        ).alias("datas_contratacao_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_birth_date"),
                1
            ).otherwise(0)
        ).alias("datas_nascimento_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_experience"),
                1
            ).otherwise(0)
        ).alias("experiencias_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_termination_date"),
                1
            ).otherwise(0)
        ).alias("datas_desligamento_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_license"),
                1
            ).otherwise(0)
        ).alias("licencas_invalidas")
    )
)

##Validar categorias da tabela drivers

In [0]:
display(
    df_drivers_silver
    .groupBy("employment_status")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_drivers_silver
    .groupBy("cdl_class")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_drivers_silver
    .groupBy("license_state")
    .count()
    .orderBy(F.desc("count"))
)

##Ler tabela trips Silver para validar relacionamento

In [0]:
df_trips_silver_ref = spark.table(
    "workspace.silver_logistics.trips"
)

##Validar relacionamento de drivers com trips

In [0]:
trips_com_driver_inexistente = (
    df_trips_silver_ref
    .filter(F.col("driver_id").isNotNull())
    .join(
        df_drivers_silver.select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
)

print(
    "Trips com driver_id não encontrado em drivers: "
    f"{trips_com_driver_inexistente.count()}"
)

##Validar motoristas sem viagem

In [0]:
drivers_sem_trip_silver = (
    df_drivers_silver
    .join(
        df_trips_silver_ref
        .filter(F.col("driver_id").isNotNull())
        .select("driver_id")
        .distinct(),
        on="driver_id",
        how="left_anti"
    )
)

display(
    drivers_sem_trip_silver
    .groupBy("employment_status")
    .count()
    .orderBy(F.desc("count"))
)

##Visualizar resultado final da tabela drivers Silver

In [0]:
display(
    df_drivers_silver.select(
        "driver_id",
        "first_name",
        "last_name",
        "hire_date",
        "termination_date",
        "hire_year_month",
        "termination_year",
        "date_of_birth",
        "driver_age",
        "years_experience",
        "tenure_years",
        "license_number",
        "license_state",
        "cdl_class",
        "home_terminal",
        "employment_status",
        "is_active_driver",
        "has_valid_hire_date",
        "has_valid_birth_date",
        "has_valid_experience",
        "has_valid_termination_date",
        "has_valid_license"
    ).limit(20)
)

##Gravar tabela drivers na camada Silver

In [0]:
(
    df_drivers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver_logistics.drivers")
)

##Validar gravação da tabela drivers Silver

In [0]:
df_drivers_silver_gravada = spark.table(
    "workspace.silver_logistics.drivers"
)

print(
    "Total de registros gravados na Silver: "
    f"{df_drivers_silver_gravada.count()}"
)

df_drivers_silver_gravada.printSchema()

##Validar estrutura persistida da tabela drivers Silver

In [0]:
display(
    df_drivers_silver_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("driver_id").alias("driver_ids_unicos"),

        F.sum(
            F.when(
                F.col("driver_id").isNull(),
                1
            ).otherwise(0)
        ).alias("driver_ids_nulos"),

        F.sum(
            F.when(
                F.col("is_active_driver"),
                1
            ).otherwise(0)
        ).alias("motoristas_ativos"),

        F.sum(
            F.when(
                ~F.col("has_valid_license"),
                1
            ).otherwise(0)
        ).alias("licencas_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_termination_date"),
                1
            ).otherwise(0)
        ).alias("datas_desligamento_invalidas")
    )
)

##Transformação da tabela trucks para a camada Silver

##Ler tabela trucks da camada Bronze

In [0]:
df_trucks_bronze = spark.table(
    "workspace.bronze_logistics.trucks"
)

##Visualizar dados da tabela trucks

In [0]:
display(
    df_trucks_bronze.limit(10)
)

##Verificar schema da tabela trucks

In [0]:
df_trucks_bronze.printSchema()

##Validar volume e chave primária da tabela trucks Bronze

In [0]:
display(
    df_trucks_bronze.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("truck_id").alias("truck_ids_unicos"),
        F.sum(
            F.when(
                F.col("truck_id").isNull(),
                1
            ).otherwise(0)
        ).alias("truck_ids_nulos")
    )
)

##Verificar valores nulos por coluna

In [0]:
display(
    df_trucks_bronze.select(
        [
            F.sum(
                F.when(
                    F.col(coluna).isNull(),
                    1
                ).otherwise(0)
            ).alias(coluna)
            for coluna in df_trucks_bronze.columns
        ]
    )
)

##Padronizar colunas textuais da tabela trucks

In [0]:
df_trucks_padronizado = (
    df_trucks_bronze
    .withColumn(
        "truck_id",
        F.trim(F.col("truck_id"))
    )
    .withColumn(
        "make",
        F.initcap(F.trim(F.col("make")))
    )
    .withColumn(
        "vin",
        F.upper(F.trim(F.col("vin")))
    )
    .withColumn(
        "fuel_type",
        F.initcap(F.trim(F.col("fuel_type")))
    )
    .withColumn(
        "status",
        F.initcap(F.trim(F.col("status")))
    )
    .withColumn(
        "home_terminal",
        F.initcap(F.trim(F.col("home_terminal")))
    )
)

##Criar atributos temporais e de idade da frota

In [0]:
df_trucks_temporal = (
    df_trucks_padronizado
    .withColumn(
        "acquisition_year",
        F.year(F.col("acquisition_date"))
    )
    .withColumn(
        "acquisition_month",
        F.month(F.col("acquisition_date"))
    )
    .withColumn(
        "acquisition_quarter",
        F.quarter(F.col("acquisition_date"))
    )
    .withColumn(
        "acquisition_year_month",
        F.date_format(
            F.col("acquisition_date"),
            "yyyy-MM"
        )
    )
    .withColumn(
        "truck_age_years",
        F.year(F.current_date()) - F.col("model_year")
    )
    .withColumn(
        "years_in_fleet",
        F.round(
            F.months_between(
                F.current_date(),
                F.col("acquisition_date")
            ) / 12,
            2
        )
    )
)

##Criar indicadores operacionais e de qualidade

In [0]:
df_trucks_silver = (
    df_trucks_temporal
    .withColumn(
        "is_active_truck",
        F.col("status") == "Active"
    )
    .withColumn(
        "has_valid_model_year",
        F.col("model_year").isNotNull()
        & (F.col("model_year") >= 1900)
        & (F.col("model_year") <= F.year(F.current_date()) + 1)
    )
    .withColumn(
        "has_valid_acquisition_date",
        F.col("acquisition_date").isNotNull()
        & (F.col("acquisition_date") <= F.current_date())
    )
    .withColumn(
        "has_valid_acquisition_mileage",
        F.col("acquisition_mileage").isNotNull()
        & (F.col("acquisition_mileage") >= 0)
    )
    .withColumn(
        "has_valid_tank_capacity",
        F.col("tank_capacity_gallons").isNotNull()
        & (F.col("tank_capacity_gallons") > 0)
    )
    .withColumn(
        "has_valid_vin",
        F.col("vin").isNotNull()
        & (F.length(F.trim(F.col("vin"))) > 0)
    )
    .withColumn(
        "has_valid_unit_number",
        F.col("unit_number").isNotNull()
        & (F.col("unit_number") > 0)
    )
    .withColumn(
        "has_complete_truck_record",
        F.col("truck_id").isNotNull()
        & F.col("unit_number").isNotNull()
        & F.col("make").isNotNull()
        & F.col("model_year").isNotNull()
        & F.col("vin").isNotNull()
        & F.col("fuel_type").isNotNull()
        & F.col("status").isNotNull()
        & F.col("home_terminal").isNotNull()
    )
)

##Validar preservação do volume

In [0]:
total_trucks_bronze = df_trucks_bronze.count()
total_trucks_silver = df_trucks_silver.count()

print(f"Total de registros na Bronze: {total_trucks_bronze}")
print(f"Total de registros na Silver: {total_trucks_silver}")
print(
    "Diferença de registros: "
    f"{total_trucks_silver - total_trucks_bronze}"
)

##Validar chave primária da tabela trucks Silver

In [0]:
display(
    df_trucks_silver.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("truck_id").alias("truck_ids_unicos"),
        F.sum(
            F.when(
                F.col("truck_id").isNull(),
                1
            ).otherwise(0)
        ).alias("truck_ids_nulos")
    )
)

##Validar duplicidades de VIN e número da unidade

In [0]:
display(
    df_trucks_silver
    .filter(F.col("vin").isNotNull())
    .groupBy("vin")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(
    df_trucks_silver
    .filter(F.col("unit_number").isNotNull())
    .groupBy("unit_number")
    .count()
    .filter(F.col("count") > 1)
)

##Validar idade e tempo de permanência da frota

In [0]:
display(
    df_trucks_silver.select(
        F.sum(
            F.when(
                F.col("truck_age_years") < 0,
                1
            ).otherwise(0)
        ).alias("idades_negativas"),

        F.sum(
            F.when(
                F.col("truck_age_years") > 40,
                1
            ).otherwise(0)
        ).alias("caminhoes_com_mais_de_40_anos"),

        F.sum(
            F.when(
                F.col("years_in_fleet") < 0,
                1
            ).otherwise(0)
        ).alias("tempos_na_frota_negativos"),

        F.sum(
            F.when(
                F.col("acquisition_mileage") < 0,
                1
            ).otherwise(0)
        ).alias("quilometragens_aquisicao_negativas")
    )
)

##Validar coerência entre ano do modelo e aquisição

In [0]:
display(
    df_trucks_silver.select(
        F.sum(
            F.when(
                F.year(F.col("acquisition_date"))
                < F.col("model_year"),
                1
            ).otherwise(0)
        ).alias("aquisicoes_anteriores_ao_ano_modelo")
    )
)

##Validar indicadores de qualidade

In [0]:
display(
    df_trucks_silver.select(
        F.sum(
            F.when(
                F.col("is_active_truck"),
                1
            ).otherwise(0)
        ).alias("caminhoes_ativos"),

        F.sum(
            F.when(
                ~F.col("has_valid_model_year"),
                1
            ).otherwise(0)
        ).alias("anos_modelo_invalidos"),

        F.sum(
            F.when(
                ~F.col("has_valid_acquisition_date"),
                1
            ).otherwise(0)
        ).alias("datas_aquisicao_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_acquisition_mileage"),
                1
            ).otherwise(0)
        ).alias("quilometragens_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_tank_capacity"),
                1
            ).otherwise(0)
        ).alias("capacidades_tanque_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_vin"),
                1
            ).otherwise(0)
        ).alias("vins_invalidos"),

        F.sum(
            F.when(
                ~F.col("has_valid_unit_number"),
                1
            ).otherwise(0)
        ).alias("numeros_unidade_invalidos"),

        F.sum(
            F.when(
                ~F.col("has_complete_truck_record"),
                1
            ).otherwise(0)
        ).alias("registros_incompletos")
    )
)

##Validar categorias de status

In [0]:
display(
    df_trucks_silver
    .groupBy("status")
    .count()
    .orderBy(F.desc("count"))
)

##Validar categorias de combustível

In [0]:
display(
    df_trucks_silver
    .groupBy("fuel_type")
    .count()
    .orderBy(F.desc("count"))
)

##Validar fabricantes da frota

In [0]:
display(
    df_trucks_silver
    .groupBy("make")
    .count()
    .orderBy(F.desc("count"))
)

##Ler trips Silver para validar o relacionamento

In [0]:
df_trips_silver_ref = spark.table(
    "workspace.silver_logistics.trips"
)

##Validar caminhões utilizados nas viagens

In [0]:
trips_com_truck_inexistente = (
    df_trips_silver_ref
    .filter(F.col("truck_id").isNotNull())
    .join(
        df_trucks_silver.select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
)

print(
    "Trips com truck_id não encontrado em trucks: "
    f"{trips_com_truck_inexistente.count()}"
)

##Identificar caminhões sem viagens

In [0]:
trucks_sem_trip = (
    df_trucks_silver
    .join(
        df_trips_silver_ref
        .filter(F.col("truck_id").isNotNull())
        .select("truck_id")
        .distinct(),
        on="truck_id",
        how="left_anti"
    )
)

print(
    "Quantidade de caminhões sem viagens: "
    f"{trucks_sem_trip.count()}"
)

display(
    trucks_sem_trip.select(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "status",
        "home_terminal"
    )
)

##Visualizar resultado final da tabela trucks Silver

In [0]:
display(
    df_trucks_silver.select(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "truck_age_years",
        "vin",
        "acquisition_date",
        "acquisition_year_month",
        "acquisition_mileage",
        "years_in_fleet",
        "fuel_type",
        "tank_capacity_gallons",
        "status",
        "home_terminal",
        "is_active_truck",
        "has_valid_model_year",
        "has_valid_acquisition_date",
        "has_valid_acquisition_mileage",
        "has_valid_tank_capacity",
        "has_valid_vin",
        "has_valid_unit_number",
        "has_complete_truck_record"
    ).limit(20)
)

##Gravar tabela trucks na camada Silver

In [0]:
(
    df_trucks_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver_logistics.trucks")
)

##Validar gravação da tabela trucks Silver

In [0]:
df_trucks_silver_gravada = spark.table(
    "workspace.silver_logistics.trucks"
)

print(
    "Total de registros gravados na Silver: "
    f"{df_trucks_silver_gravada.count()}"
)

df_trucks_silver_gravada.printSchema()

##Validar estrutura persistida

In [0]:
display(
    df_trucks_silver_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("truck_id").alias("truck_ids_unicos"),

        F.sum(
            F.when(
                F.col("truck_id").isNull(),
                1
            ).otherwise(0)
        ).alias("truck_ids_nulos"),

        F.sum(
            F.when(
                F.col("is_active_truck"),
                1
            ).otherwise(0)
        ).alias("caminhoes_ativos"),

        F.sum(
            F.when(
                ~F.col("has_valid_vin"),
                1
            ).otherwise(0)
        ).alias("vins_invalidos"),

        F.sum(
            F.when(
                ~F.col("has_complete_truck_record"),
                1
            ).otherwise(0)
        ).alias("registros_incompletos")
    )
)

##Transformação da tabela delivery_events para a camada Silver

##Ler tabela delivery_events da camada Bronze

In [0]:
df_delivery_events_bronze = spark.table(
    "workspace.bronze_logistics.delivery_events"
)

##Visualizar os registros

In [0]:
display(
    df_delivery_events_bronze.limit(10)
)

##Verificar schema

In [0]:
df_delivery_events_bronze.printSchema()

##Validar volume e chave primária

In [0]:
print(df_delivery_events_bronze.columns)

In [0]:
display(
    df_delivery_events_bronze.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("event_id").alias("event_ids_unicos"),
        F.sum(
            F.when(
                F.col("event_id").isNull(),
                1
            ).otherwise(0)
        ).alias("event_ids_nulos")
    )
)

##Verificar valores nulos por coluna

In [0]:
display(
    df_delivery_events_bronze.select(
        [
            F.sum(
                F.when(
                    F.col(coluna).isNull(),
                    1
                ).otherwise(0)
            ).alias(coluna)
            for coluna in df_delivery_events_bronze.columns
        ]
    )
)

##Padronizar colunas textuais da tabela delivery_events

In [0]:
df_delivery_events_padronizado = (
    df_delivery_events_bronze
    .withColumn(
        "event_id",
        F.trim(F.col("event_id"))
    )
    .withColumn(
        "load_id",
        F.trim(F.col("load_id"))
    )
    .withColumn(
        "trip_id",
        F.trim(F.col("trip_id"))
    )
    .withColumn(
        "event_type",
        F.initcap(F.trim(F.col("event_type")))
    )
    .withColumn(
        "facility_id",
        F.trim(F.col("facility_id"))
    )
    .withColumn(
        "location_city",
        F.initcap(F.trim(F.col("location_city")))
    )
    .withColumn(
        "location_state",
        F.upper(F.trim(F.col("location_state")))
    )
)

##Criar atributos temporais e localização

In [0]:
df_delivery_events_temporal = (
    df_delivery_events_padronizado
    .withColumn(
        "scheduled_date",
        F.to_date(F.col("scheduled_datetime"))
    )
    .withColumn(
        "actual_date",
        F.to_date(F.col("actual_datetime"))
    )
    .withColumn(
        "scheduled_year",
        F.year(F.col("scheduled_datetime"))
    )
    .withColumn(
        "scheduled_month",
        F.month(F.col("scheduled_datetime"))
    )
    .withColumn(
        "scheduled_quarter",
        F.quarter(F.col("scheduled_datetime"))
    )
    .withColumn(
        "scheduled_year_month",
        F.date_format(
            F.col("scheduled_datetime"),
            "yyyy-MM"
        )
    )
    .withColumn(
        "scheduled_hour",
        F.hour(F.col("scheduled_datetime"))
    )
    .withColumn(
        "location",
        F.concat_ws(
            " - ",
            F.col("location_city"),
            F.col("location_state")
        )
    )
)

##Criar métricas de antecipação e atraso

In [0]:
df_delivery_events_metricas = (
    df_delivery_events_temporal
    .withColumn(
        "event_variance_minutes",
        F.round(
            (
                F.unix_timestamp(F.col("actual_datetime"))
                - F.unix_timestamp(F.col("scheduled_datetime"))
            ) / 60,
            2
        )
    )
    .withColumn(
        "delay_minutes",
        F.when(
            F.col("event_variance_minutes") > 0,
            F.col("event_variance_minutes")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "early_minutes",
        F.when(
            F.col("event_variance_minutes") < 0,
            F.abs(F.col("event_variance_minutes"))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "is_on_time_calculated",
        F.col("actual_datetime")
        <= F.col("scheduled_datetime")
    )
)

##Criar indicadores operacionais e de qualidade

In [0]:
df_delivery_events_silver = (
    df_delivery_events_metricas
    .withColumn(
        "is_pickup_event",
        F.col("event_type") == "Pickup"
    )
    .withColumn(
        "is_delivery_event",
        F.col("event_type") == "Delivery"
    )
    .withColumn(
        "is_delayed_event",
        F.col("delay_minutes") > 0
    )
    .withColumn(
        "has_detention",
        F.col("detention_minutes") > 0
    )
    .withColumn(
        "on_time_flag_consistent",
        F.col("on_time_flag")
        == F.col("is_on_time_calculated")
    )
    .withColumn(
        "has_valid_scheduled_datetime",
        F.col("scheduled_datetime").isNotNull()
    )
    .withColumn(
        "has_valid_actual_datetime",
        F.col("actual_datetime").isNotNull()
    )
    .withColumn(
        "has_valid_detention_minutes",
        F.col("detention_minutes").isNotNull()
        & (F.col("detention_minutes") >= 0)
    )
    .withColumn(
        "has_complete_relationship",
        F.col("load_id").isNotNull()
        & F.col("trip_id").isNotNull()
        & F.col("facility_id").isNotNull()
    )
    .withColumn(
        "has_complete_location",
        F.col("location_city").isNotNull()
        & F.col("location_state").isNotNull()
    )
)

##Validar preservação do volume

In [0]:
total_delivery_events_bronze = df_delivery_events_bronze.count()
total_delivery_events_silver = df_delivery_events_silver.count()

print(
    f"Total de registros na Bronze: "
    f"{total_delivery_events_bronze}"
)

print(
    f"Total de registros na Silver: "
    f"{total_delivery_events_silver}"
)

print(
    "Diferença de registros: "
    f"{total_delivery_events_silver - total_delivery_events_bronze}"
)

##Validar chave primária

In [0]:
display(
    df_delivery_events_silver.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("event_id").alias(
            "event_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("event_id").isNull(),
                1
            ).otherwise(0)
        ).alias("event_ids_nulos")
    )
)

##Validar métricas de tempo

In [0]:
display(
    df_delivery_events_silver.select(
        F.sum(
            F.when(
                F.col("event_variance_minutes").isNull(),
                1
            ).otherwise(0)
        ).alias("variacoes_tempo_nulas"),

        F.sum(
            F.when(
                F.col("delay_minutes") < 0,
                1
            ).otherwise(0)
        ).alias("atrasos_negativos"),

        F.sum(
            F.when(
                F.col("early_minutes") < 0,
                1
            ).otherwise(0)
        ).alias("antecipacoes_negativas"),

        F.sum(
            F.when(
                F.col("detention_minutes") < 0,
                1
            ).otherwise(0)
        ).alias("detencoes_negativas")
    )
)

##Validar consistência do on_time_flag

In [0]:
display(
    df_delivery_events_silver.select(
        F.sum(
            F.when(
                F.col("on_time_flag_consistent"),
                1
            ).otherwise(0)
        ).alias("flags_consistentes"),

        F.sum(
            F.when(
                ~F.col("on_time_flag_consistent"),
                1
            ).otherwise(0)
        ).alias("flags_inconsistentes")
    )
)

##Validar indicadores de qualidade

In [0]:
display(
    df_delivery_events_silver.select(
        F.sum(
            F.when(
                ~F.col("has_valid_scheduled_datetime"),
                1
            ).otherwise(0)
        ).alias("datas_programadas_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_actual_datetime"),
                1
            ).otherwise(0)
        ).alias("datas_reais_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_detention_minutes"),
                1
            ).otherwise(0)
        ).alias("tempos_detencao_invalidos"),

        F.sum(
            F.when(
                ~F.col("has_complete_relationship"),
                1
            ).otherwise(0)
        ).alias("relacionamentos_incompletos"),

        F.sum(
            F.when(
                ~F.col("has_complete_location"),
                1
            ).otherwise(0)
        ).alias("localizacoes_incompletas")
    )
)

##Validar categorias de eventos

In [0]:
display(
    df_delivery_events_silver
    .groupBy("event_type")
    .count()
    .orderBy(F.desc("count"))
)

##Analisar eventos pontuais e atrasados

In [0]:
display(
    df_delivery_events_silver.select(
        F.count("*").alias("total_eventos"),

        F.sum(
            F.when(
                F.col("is_on_time_calculated"),
                1
            ).otherwise(0)
        ).alias("eventos_no_prazo"),

        F.sum(
            F.when(
                F.col("is_delayed_event"),
                1
            ).otherwise(0)
        ).alias("eventos_atrasados"),

        F.sum(
            F.when(
                F.col("has_detention"),
                1
            ).otherwise(0)
        ).alias("eventos_com_detencao"),

        F.round(
            F.avg("delay_minutes"),
            2
        ).alias("media_atraso_minutos"),

        F.round(
            F.avg("detention_minutes"),
            2
        ).alias("media_detencao_minutos")
    )
)

##Validar relacionamento com loads

In [0]:
df_loads_silver_ref = spark.table(
    "workspace.silver_logistics.loads"
)

events_com_load_inexistente = (
    df_delivery_events_silver
    .join(
        df_loads_silver_ref.select("load_id"),
        on="load_id",
        how="left_anti"
    )
)

print(
    "Eventos com load_id não encontrado em loads: "
    f"{events_com_load_inexistente.count()}"
)

##Validar relacionamento com trips

In [0]:
df_trips_silver_ref = spark.table(
    "workspace.silver_logistics.trips"
)

events_com_trip_inexistente = (
    df_delivery_events_silver
    .join(
        df_trips_silver_ref.select("trip_id"),
        on="trip_id",
        how="left_anti"
    )
)

print(
    "Eventos com trip_id não encontrado em trips: "
    f"{events_com_trip_inexistente.count()}"
)

##Validar quantidade de eventos por viagem

In [0]:
eventos_por_trip = (
    df_delivery_events_silver
    .groupBy("trip_id")
    .agg(
        F.count("*").alias("quantidade_eventos"),
        F.countDistinct("event_type").alias(
            "tipos_eventos_distintos"
        )
    )
)

In [0]:
display(
    eventos_por_trip.groupBy(
        "quantidade_eventos",
        "tipos_eventos_distintos"
    )
    .count()
    .orderBy(
        "quantidade_eventos",
        "tipos_eventos_distintos"
    )
)

##Validar viagens sem dois eventos

In [0]:
display(
    eventos_por_trip.filter(
        (F.col("quantidade_eventos") != 2)
        | (F.col("tipos_eventos_distintos") != 2)
    )
)

##Visualizar resultado final

In [0]:
display(
    df_delivery_events_silver.select(
        "event_id",
        "load_id",
        "trip_id",
        "event_type",
        "facility_id",
        "location",
        "scheduled_datetime",
        "actual_datetime",
        "event_variance_minutes",
        "delay_minutes",
        "early_minutes",
        "detention_minutes",
        "on_time_flag",
        "is_on_time_calculated",
        "on_time_flag_consistent",
        "is_pickup_event",
        "is_delivery_event",
        "is_delayed_event",
        "has_detention",
        "has_complete_relationship",
        "has_complete_location"
    ).limit(20)
)

##Gravar tabela delivery_events na camada Silver

In [0]:
(
    df_delivery_events_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.silver_logistics.delivery_events"
    )
)

##Validar gravação

In [0]:
df_delivery_events_silver_gravada = spark.table(
    "workspace.silver_logistics.delivery_events"
)

print(
    "Total de registros gravados na Silver: "
    f"{df_delivery_events_silver_gravada.count()}"
)

df_delivery_events_silver_gravada.printSchema()

##Validar estrutura persistida

In [0]:
display(
    df_delivery_events_silver_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("event_id").alias(
            "event_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("event_id").isNull(),
                1
            ).otherwise(0)
        ).alias("event_ids_nulos"),

        F.sum(
            F.when(
                F.col("is_on_time_calculated"),
                1
            ).otherwise(0)
        ).alias("eventos_no_prazo"),

        F.sum(
            F.when(
                F.col("is_delayed_event"),
                1
            ).otherwise(0)
        ).alias("eventos_atrasados"),

        F.sum(
            F.when(
                ~F.col("on_time_flag_consistent"),
                1
            ).otherwise(0)
        ).alias("flags_inconsistentes"),

        F.sum(
            F.when(
                ~F.col("has_complete_relationship"),
                1
            ).otherwise(0)
        ).alias("relacionamentos_incompletos")
    )
)

##Transformação da tabela delivery_events para a camada Silver

##Ler tabela fuel_purchases da camada Bronze

In [0]:
df_fuel_purchases_bronze = spark.table(
    "workspace.bronze_logistics.fuel_purchases"
)

##Visualizar registros da tabela fuel_purchases

In [0]:
display(
    df_fuel_purchases_bronze.limit(10)
)

##Verificar schema da tabela fuel_purchases

In [0]:
df_fuel_purchases_bronze.printSchema()

##Validar volume e chave primária

In [0]:
print(df_fuel_purchases_bronze.columns)

In [0]:
display(
    df_fuel_purchases_bronze.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("fuel_purchase_id").alias(
            "fuel_purchase_ids_unicos"
        ),
        F.sum(
            F.when(
                F.col("fuel_purchase_id").isNull(),
                1
            ).otherwise(0)
        ).alias("fuel_purchase_ids_nulos")
    )
)

##Verificar valores nulos por coluna

In [0]:
display(
    df_fuel_purchases_bronze.select(
        [
            F.sum(
                F.when(
                    F.col(coluna).isNull(),
                    1
                ).otherwise(0)
            ).alias(coluna)
            for coluna in df_fuel_purchases_bronze.columns
        ]
    )
)

##Padronizar colunas textuais da tabela fuel_purchases

In [0]:
df_fuel_purchases_padronizado = (
    df_fuel_purchases_bronze
    .withColumn(
        "fuel_purchase_id",
        F.trim(F.col("fuel_purchase_id"))
    )
    .withColumn(
        "trip_id",
        F.trim(F.col("trip_id"))
    )
    .withColumn(
        "truck_id",
        F.trim(F.col("truck_id"))
    )
    .withColumn(
        "driver_id",
        F.trim(F.col("driver_id"))
    )
    .withColumn(
        "location_city",
        F.initcap(F.trim(F.col("location_city")))
    )
    .withColumn(
        "location_state",
        F.upper(F.trim(F.col("location_state")))
    )
    .withColumn(
        "fuel_card_number",
        F.upper(F.trim(F.col("fuel_card_number")))
    )
)

##Criar atributos temporais e de localização

In [0]:
df_fuel_purchases_temporal = (
    df_fuel_purchases_padronizado
    .withColumn(
        "purchase_day",
        F.to_date(F.col("purchase_date"))
    )
    .withColumn(
        "purchase_year",
        F.year(F.col("purchase_date"))
    )
    .withColumn(
        "purchase_month",
        F.month(F.col("purchase_date"))
    )
    .withColumn(
        "purchase_quarter",
        F.quarter(F.col("purchase_date"))
    )
    .withColumn(
        "purchase_year_month",
        F.date_format(
            F.col("purchase_date"),
            "yyyy-MM"
        )
    )
    .withColumn(
        "purchase_hour",
        F.hour(F.col("purchase_date"))
    )
    .withColumn(
        "purchase_location",
        F.concat_ws(
            " - ",
            F.col("location_city"),
            F.col("location_state")
        )
    )
)

##Criar métricas financeiras da compra

In [0]:
df_fuel_purchases_metricas = (
    df_fuel_purchases_temporal
    .withColumn(
        "calculated_total_cost",
        F.round(
            F.col("gallons")
            * F.col("price_per_gallon"),
            2
        )
    )
    .withColumn(
        "total_cost_difference",
        F.round(
            F.col("total_cost")
            - F.col("calculated_total_cost"),
            2
        )
    )
    .withColumn(
        "cost_per_gallon_calculated",
        F.when(
            F.col("gallons") > 0,
            F.round(
                F.col("total_cost") / F.col("gallons"),
                3
            )
        )
    )
)

##Criar indicadores de qualidade e completude

In [0]:
df_fuel_purchases_silver = (
    df_fuel_purchases_metricas
    .withColumn(
        "has_truck",
        F.col("truck_id").isNotNull()
    )
    .withColumn(
        "has_driver",
        F.col("driver_id").isNotNull()
    )
    .withColumn(
        "has_complete_resource_assignment",
        F.col("truck_id").isNotNull()
        & F.col("driver_id").isNotNull()
    )
    .withColumn(
        "has_valid_purchase_date",
        F.col("purchase_date").isNotNull()
        & (F.col("purchase_date") <= F.current_timestamp())
    )
    .withColumn(
        "has_valid_gallons",
        F.col("gallons").isNotNull()
        & (F.col("gallons") > 0)
    )
    .withColumn(
        "has_valid_price_per_gallon",
        F.col("price_per_gallon").isNotNull()
        & (F.col("price_per_gallon") > 0)
    )
    .withColumn(
        "has_valid_total_cost",
        F.col("total_cost").isNotNull()
        & (F.col("total_cost") > 0)
    )
    .withColumn(
        "has_consistent_total_cost",
        F.abs(F.col("total_cost_difference")) <= 0.02
    )
    .withColumn(
        "has_complete_location",
        F.col("location_city").isNotNull()
        & F.col("location_state").isNotNull()
    )
    .withColumn(
        "has_valid_fuel_card",
        F.col("fuel_card_number").isNotNull()
        & (F.length(F.trim(F.col("fuel_card_number"))) > 0)
    )
)

##Validar preservação do volume

In [0]:
total_fuel_purchases_bronze = (
    df_fuel_purchases_bronze.count()
)

total_fuel_purchases_silver = (
    df_fuel_purchases_silver.count()
)

print(
    f"Total de registros na Bronze: "
    f"{total_fuel_purchases_bronze}"
)

print(
    f"Total de registros na Silver: "
    f"{total_fuel_purchases_silver}"
)

print(
    "Diferença de registros: "
    f"{total_fuel_purchases_silver - total_fuel_purchases_bronze}"
)

##Validar chave primária

In [0]:
display(
    df_fuel_purchases_silver.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("fuel_purchase_id").alias(
            "fuel_purchase_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("fuel_purchase_id").isNull(),
                1
            ).otherwise(0)
        ).alias("fuel_purchase_ids_nulos")
    )
)

##Validar duplicidade do cartão de combustível

In [0]:
display(
    df_fuel_purchases_silver.select(
        F.countDistinct("fuel_card_number").alias(
            "cartoes_combustivel_distintos"
        ),
        F.count("*").alias("total_compras")
    )
)

##Validar métricas financeiras

In [0]:
display(
    df_fuel_purchases_silver.select(
        F.sum(
            F.when(
                F.col("gallons") <= 0,
                1
            ).otherwise(0)
        ).alias("quantidades_galoes_invalidas"),

        F.sum(
            F.when(
                F.col("price_per_gallon") <= 0,
                1
            ).otherwise(0)
        ).alias("precos_por_galao_invalidos"),

        F.sum(
            F.when(
                F.col("total_cost") <= 0,
                1
            ).otherwise(0)
        ).alias("custos_totais_invalidos"),

        F.sum(
            F.when(
                F.col("calculated_total_cost").isNull(),
                1
            ).otherwise(0)
        ).alias("custos_calculados_nulos"),

        F.sum(
            F.when(
                F.col("cost_per_gallon_calculated").isNull(),
                1
            ).otherwise(0)
        ).alias("custos_por_galao_calculados_nulos")
    )
)

##Validar consistência do custo total

In [0]:
display(
    df_fuel_purchases_silver.select(
        F.sum(
            F.when(
                F.col("has_consistent_total_cost"),
                1
            ).otherwise(0)
        ).alias("custos_totais_consistentes"),

        F.sum(
            F.when(
                ~F.col("has_consistent_total_cost"),
                1
            ).otherwise(0)
        ).alias("custos_totais_inconsistentes"),

        F.round(
            F.max(F.abs(F.col("total_cost_difference"))),
            2
        ).alias("maior_diferenca_encontrada")
    )
)

##Visualizar compras com custo inconsistente

In [0]:
display(
    df_fuel_purchases_silver
    .filter(
        ~F.col("has_consistent_total_cost")
    )
    .select(
        "fuel_purchase_id",
        "gallons",
        "price_per_gallon",
        "total_cost",
        "calculated_total_cost",
        "total_cost_difference"
    )
    .orderBy(
        F.desc(
            F.abs(F.col("total_cost_difference"))
        )
    )
    .limit(20)
)

##Validar indicadores de qualidade

In [0]:
display(
    df_fuel_purchases_silver.select(
        F.sum(
            F.when(
                ~F.col("has_valid_purchase_date"),
                1
            ).otherwise(0)
        ).alias("datas_compra_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_gallons"),
                1
            ).otherwise(0)
        ).alias("quantidades_galoes_invalidas"),

        F.sum(
            F.when(
                ~F.col("has_valid_price_per_gallon"),
                1
            ).otherwise(0)
        ).alias("precos_invalidos"),

        F.sum(
            F.when(
                ~F.col("has_valid_total_cost"),
                1
            ).otherwise(0)
        ).alias("custos_invalidos"),

        F.sum(
            F.when(
                ~F.col("has_complete_location"),
                1
            ).otherwise(0)
        ).alias("localizacoes_incompletas"),

        F.sum(
            F.when(
                ~F.col("has_valid_fuel_card"),
                1
            ).otherwise(0)
        ).alias("cartoes_combustivel_invalidos")
    )
)

##Validar preenchimento dos recursos

In [0]:
display(
    df_fuel_purchases_silver.select(
        F.sum(
            F.when(
                F.col("has_truck"),
                1
            ).otherwise(0)
        ).alias("compras_com_caminhao"),

        F.sum(
            F.when(
                ~F.col("has_truck"),
                1
            ).otherwise(0)
        ).alias("compras_sem_caminhao"),

        F.sum(
            F.when(
                F.col("has_driver"),
                1
            ).otherwise(0)
        ).alias("compras_com_motorista"),

        F.sum(
            F.when(
                ~F.col("has_driver"),
                1
            ).otherwise(0)
        ).alias("compras_sem_motorista"),

        F.sum(
            F.when(
                F.col("has_complete_resource_assignment"),
                1
            ).otherwise(0)
        ).alias("compras_com_recursos_completos")
    )
)

##Ler tabelas Silver de referência

In [0]:
df_trips_silver_ref = spark.table(
    "workspace.silver_logistics.trips"
)

df_trucks_silver_ref = spark.table(
    "workspace.silver_logistics.trucks"
)

df_drivers_silver_ref = spark.table(
    "workspace.silver_logistics.drivers"
)

##Validar relacionamento com trips

In [0]:
fuel_com_trip_inexistente = (
    df_fuel_purchases_silver
    .join(
        df_trips_silver_ref.select("trip_id"),
        on="trip_id",
        how="left_anti"
    )
)

print(
    "Compras com trip_id não encontrado em trips: "
    f"{fuel_com_trip_inexistente.count()}"
)

##Validar relacionamento com trucks

In [0]:
fuel_com_truck_inexistente = (
    df_fuel_purchases_silver
    .filter(
        F.col("truck_id").isNotNull()
    )
    .join(
        df_trucks_silver_ref.select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
)

print(
    "Compras com truck_id preenchido e não encontrado: "
    f"{fuel_com_truck_inexistente.count()}"
)

##Validar relacionamento com drivers

In [0]:
fuel_com_driver_inexistente = (
    df_fuel_purchases_silver
    .filter(
        F.col("driver_id").isNotNull()
    )
    .join(
        df_drivers_silver_ref.select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
)

print(
    "Compras com driver_id preenchido e não encontrado: "
    f"{fuel_com_driver_inexistente.count()}"
)

##Comparar caminhão da compra com caminhão da viagem

In [0]:
df_fuel_com_trip = (
    df_fuel_purchases_silver.alias("fuel")
    .join(
        df_trips_silver_ref.select(
            "trip_id",
            F.col("truck_id").alias("trip_truck_id"),
            F.col("driver_id").alias("trip_driver_id")
        ).alias("trip"),
        on="trip_id",
        how="left"
    )
)

In [0]:
display(
    df_fuel_com_trip.select(
        F.sum(
            F.when(
                F.col("fuel.truck_id").isNotNull()
                & F.col("trip_truck_id").isNotNull()
                & (
                    F.col("fuel.truck_id")
                    != F.col("trip_truck_id")
                ),
                1
            ).otherwise(0)
        ).alias("caminhoes_divergentes_da_viagem"),

        F.sum(
            F.when(
                F.col("fuel.driver_id").isNotNull()
                & F.col("trip_driver_id").isNotNull()
                & (
                    F.col("fuel.driver_id")
                    != F.col("trip_driver_id")
                ),
                1
            ).otherwise(0)
        ).alias("motoristas_divergentes_da_viagem")
    )
)

##Criar indicadores de consistência com a viagem

In [0]:
df_fuel_purchases_silver_final = (
    df_fuel_com_trip
    .withColumn(
        "truck_matches_trip",
        F.when(
            F.col("fuel.truck_id").isNull()
            | F.col("trip_truck_id").isNull(),
            F.lit(None).cast("boolean")
        ).otherwise(
            F.col("fuel.truck_id")
            == F.col("trip_truck_id")
        )
    )
    .withColumn(
        "driver_matches_trip",
        F.when(
            F.col("fuel.driver_id").isNull()
            | F.col("trip_driver_id").isNull(),
            F.lit(None).cast("boolean")
        ).otherwise(
            F.col("fuel.driver_id")
            == F.col("trip_driver_id")
        )
    )
    .drop(
        "trip_truck_id",
        "trip_driver_id"
    )
)

##Validar volume após o relacionamento

In [0]:
print(
    "Registros antes do relacionamento: "
    f"{df_fuel_purchases_silver.count()}"
)

print(
    "Registros após o relacionamento: "
    f"{df_fuel_purchases_silver_final.count()}"
)

##Analisar compras por período

In [0]:
display(
    df_fuel_purchases_silver_final
    .groupBy("purchase_year")
    .agg(
        F.count("*").alias("total_compras"),
        F.round(
            F.sum("gallons"),
            2
        ).alias("total_galoes"),
        F.round(
            F.sum("total_cost"),
            2
        ).alias("custo_total")
    )
    .orderBy("purchase_year")
)

##Analisar métricas gerais de combustível

In [0]:
display(
    df_fuel_purchases_silver_final.select(
        F.count("*").alias("total_compras"),

        F.round(
            F.sum("gallons"),
            2
        ).alias("total_galoes"),

        F.round(
            F.sum("total_cost"),
            2
        ).alias("custo_total_combustivel"),

        F.round(
            F.avg("gallons"),
            2
        ).alias("media_galoes_por_compra"),

        F.round(
            F.avg("price_per_gallon"),
            3
        ).alias("preco_medio_por_galao"),

        F.round(
            F.avg("total_cost"),
            2
        ).alias("custo_medio_por_compra")
    )
)

##Visualizar resultado final

In [0]:
display(
    df_fuel_purchases_silver_final.select(
        "fuel_purchase_id",
        "trip_id",
        "truck_id",
        "driver_id",
        "purchase_date",
        "purchase_year_month",
        "purchase_location",
        "gallons",
        "price_per_gallon",
        "total_cost",
        "calculated_total_cost",
        "total_cost_difference",
        "fuel_card_number",
        "has_truck",
        "has_driver",
        "has_complete_resource_assignment",
        "has_valid_purchase_date",
        "has_valid_gallons",
        "has_valid_price_per_gallon",
        "has_valid_total_cost",
        "has_consistent_total_cost",
        "truck_matches_trip",
        "driver_matches_trip"
    ).limit(20)
)

##Gravar tabela fuel_purchases na camada Silver

In [0]:
(
    df_fuel_purchases_silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.silver_logistics.fuel_purchases"
    )
)

##Validar gravação

In [0]:
df_fuel_purchases_silver_gravada = spark.table(
    "workspace.silver_logistics.fuel_purchases"
)

print(
    "Total de registros gravados na Silver: "
    f"{df_fuel_purchases_silver_gravada.count()}"
)

df_fuel_purchases_silver_gravada.printSchema()

##Validar estrutura persistida

In [0]:
display(
    df_fuel_purchases_silver_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("fuel_purchase_id").alias(
            "fuel_purchase_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("fuel_purchase_id").isNull(),
                1
            ).otherwise(0)
        ).alias("fuel_purchase_ids_nulos"),

        F.sum(
            F.when(
                ~F.col("has_truck"),
                1
            ).otherwise(0)
        ).alias("compras_sem_caminhao"),

        F.sum(
            F.when(
                ~F.col("has_driver"),
                1
            ).otherwise(0)
        ).alias("compras_sem_motorista"),

        F.sum(
            F.when(
                ~F.col("has_consistent_total_cost"),
                1
            ).otherwise(0)
        ).alias("custos_totais_inconsistentes"),

        F.sum(
            F.when(
                F.col("truck_matches_trip") == False,
                1
            ).otherwise(0)
        ).alias("caminhoes_divergentes_da_viagem"),

        F.sum(
            F.when(
                F.col("driver_matches_trip") == False,
                1
            ).otherwise(0)
        ).alias("motoristas_divergentes_da_viagem")
    )
)